# Eval Results Viewer
Load JSON results from one or more eval directories, aggregate, and visualise by task.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# List as many eval subdirs as you like — results are merged.
STABLEWM_HOME = "/lustre/fswork/projects/rech/yil/ugy35qd/.stable_worldmodel"

EVAL_DIRS = [
    f"{STABLEWM_HOME}/eval/43341",
    # f"{STABLEWM_HOME}/eval/43500",  # add more job IDs here
]

# Optional: rename models for display. Keys = checkpoint name, values = display label.
MODEL_LABELS = {
    # "scenario3_sensor_w1_H50_v3": "JEPA w=1 H=50",
    # "scenario3_sensor_w1_H50_S5": "JEPA w=1 H=50 S=5",
}

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

plt.rcParams.update({
    "font.family": "serif", "font.size": 10,
    "axes.titlesize": 10, "axes.labelsize": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 1.8, "figure.dpi": 120,
})

COMPONENT_SHORT = {
    "deg_CmpBst_s_mapEff_in": "Bst/Eff",
    "deg_CmpBst_s_mapWc_in":  "Bst/Wc",
    "deg_CmpFan_s_mapEff_in": "Fan/Eff",
    "deg_CmpFan_s_mapWc_in":  "Fan/Wc",
    "deg_CmpH_s_mapEff_in":   "HPC/Eff",
    "deg_CmpH_s_mapWc_in":    "HPC/Wc",
    "deg_TrbH_s_mapEff_in":   "HPT/Eff",
    "deg_TrbH_s_mapWc_in":    "HPT/Wc",
    "deg_TrbL_s_mapEff_in":   "LPT/Eff",
    "deg_TrbL_s_mapWc_in":    "LPT/Wc",
}

def label(model):
    return MODEL_LABELS.get(model, model)

# ── Load all JSONs ─────────────────────────────────────────────────────────────
all_results = []
for d in EVAL_DIRS:
    for jf in sorted(Path(d).glob("*.json")):
        if jf.name == "summary.json":
            continue
        with open(jf) as f:
            res = json.load(f)
        res["_source_dir"] = str(d)
        all_results.append(res)

print(f"Loaded {len(all_results)} result(s):")
for r in all_results:
    status = "ERROR: " + r['error'][:60] if "error" in r else "ok"
    print(f"  [{status}]  {r['name']}  (obs_window={r.get('obs_window_size','?')}, params={r.get('n_params','?'):,})")

## Model overview

In [ ]:
rows = []
for r in all_results:
    if "error" in r:
        continue
    t1 = r.get("task1_hi") or {}
    # best mean R² across seq_lens
    t1_r2 = max((v.get("__mean__", {}).get("r2", float("nan")) for v in t1.values()), default=float("nan"))
    t1_sl = next((sl for sl, v in t1.items() if v.get("__mean__", {}).get("r2", -999) == t1_r2), "-")

    t2 = r.get("task2_delta_hi") or {}
    t2_r2 = max((v.get("__mean__", {}).get("r2", float("nan")) for v in t2.values()), default=float("nan"))

    alarm = (r.get("task2b_alarm") or {}).get("K20") or {}
    t3 = r.get("task3_forecast") or {}
    mr = t3.get("mean_rmse_all", [])
    gap = t3.get("mean_rmse_gap", [])

    rows.append({
        "Model": label(r["name"]),
        "obs_window": r.get("obs_window_size", "?"),
        "n_params": r.get("n_params", "?"),
        "T1 R² (best sl)": f"{t1_r2:.3f} ({t1_sl})",
        "T2 R² (best sl)": f"{t2_r2:.3f}",
        "Alarm AUC@K20": f"{alarm.get('auc', float('nan')):.3f}" if alarm.get('auc') is not None else "—",
        "T3 RMSE τ=1": f"{mr[0]:.4f}" if mr else "—",
        "T3 RMSE τ=10": f"{mr[9]:.4f}" if len(mr) > 9 else "—",
        "T3 gap τ=10": f"{gap[9]:+.4f}" if len(gap) > 9 else "—",
    })

overview = pd.DataFrame(rows).set_index("Model")
overview

## Task 1 — HI State Estimation
### Mean R² / RMSE / Pearson-r across seq_lens

In [ ]:
t1_rows = []
for r in all_results:
    if "error" in r:
        continue
    for sl_tag, dims in (r.get("task1_hi") or {}).items():
        mean = dims.get("__mean__", {})
        t1_rows.append({
            "model": label(r["name"]),
            "seq_len": sl_tag,
            "R²":        mean.get("r2",        float("nan")),
            "RMSE":      mean.get("rmse",       float("nan")),
            "Pearson-r": mean.get("pearson_r",  float("nan")),
        })

t1_df = pd.DataFrame(t1_rows)
display(t1_df.pivot(index="model", columns="seq_len", values=["R²", "RMSE", "Pearson-r"]).round(4))

In [ ]:
# Per-component R² for a chosen seq_len
SEQ_LEN = "sl1"   # change to "sl10" or "sl50" if available

comp_rows = []
for r in all_results:
    if "error" in r:
        continue
    dims = (r.get("task1_hi") or {}).get(SEQ_LEN, {})
    for comp, metrics in dims.items():
        if comp == "__mean__":
            continue
        comp_rows.append({
            "model": label(r["name"]),
            "component": COMPONENT_SHORT.get(comp, comp),
            "R²": metrics.get("r2", float("nan")),
        })

comp_df = pd.DataFrame(comp_rows)
models  = comp_df["model"].unique()
comps   = list(COMPONENT_SHORT.values())
x = np.arange(len(comps))
width = 0.8 / max(len(models), 1)

fig, ax = plt.subplots(figsize=(10, 3.5))
for i, m in enumerate(sorted(models)):
    sub = comp_df[comp_df["model"] == m].set_index("component")["R²"]
    vals = [float(sub.get(c, np.nan)) for c in comps]
    ax.bar(x + (i - len(models)/2 + 0.5) * width, vals, width, label=m, alpha=0.85)

ax.axhline(1.0, color="grey", lw=0.6, ls="--", alpha=0.5)
ax.set_xticks(x); ax.set_xticklabels(comps, rotation=35, ha="right")
ax.set_ylabel("$R^2$"); ax.set_title(f"Task 1 — Per-component R² ({SEQ_LEN})")
ax.legend(frameon=False, bbox_to_anchor=(1, 1))
plt.tight_layout(); plt.show()

## Task 2 — Degradation Velocity (ΔHI)
### Mean metrics across seq_lens

In [ ]:
t2_rows = []
for r in all_results:
    if "error" in r:
        continue
    for sl_tag, dims in (r.get("task2_delta_hi") or {}).items():
        mean = dims.get("__mean__", {})
        t2_rows.append({
            "model": label(r["name"]),
            "seq_len": sl_tag,
            "R²":        mean.get("r2",        float("nan")),
            "RMSE":      mean.get("rmse",       float("nan")),
            "Pearson-r": mean.get("pearson_r",  float("nan")),
        })

t2_df = pd.DataFrame(t2_rows)
if not t2_df.empty:
    display(t2_df.pivot(index="model", columns="seq_len", values=["R²", "RMSE", "Pearson-r"]).round(4))
else:
    print("No Task 2 results (run with --tasks 1 2 or all)")

## Task 2b — Maintenance Alarm

In [ ]:
alarm_rows = []
for r in all_results:
    if "error" in r:
        continue
    for k_str, metrics in (r.get("task2b_alarm") or {}).items():
        if not metrics:
            continue
        alarm_rows.append({
            "model":   label(r["name"]),
            "horizon": k_str,
            "AUC":     metrics.get("auc"),
            "Avg Prec": metrics.get("avg_precision"),
            "F1":      metrics.get("f1"),
        })

alarm_df = pd.DataFrame(alarm_rows)
if not alarm_df.empty:
    display(alarm_df.pivot(index="model", columns="horizon", values=["AUC", "Avg Prec", "F1"]).round(3))
else:
    print("No Task 2b results")

## Task 3 — Latent Forecasting
### RMSE(τ) curves — clean / event / gap

In [ ]:
t3_results = [(label(r["name"]), r["task3_forecast"]) for r in all_results
              if "error" not in r and r.get("task3_forecast")]

if not t3_results:
    print("No Task 3 results (run with --tasks 1 2 3 or all)")
else:
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=False)
    prop_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]

    for i, (name, t3) in enumerate(t3_results):
        col = prop_cycle[i % len(prop_cycle)]
        tau = np.arange(1, len(t3["mean_rmse_all"]) + 1)
        axes[0].plot(tau, t3["mean_rmse_clean"], color=col, label=name)
        axes[1].plot(tau, t3["mean_rmse_event"], color=col, label=name)
        axes[2].plot(tau, t3["mean_rmse_gap"],   color=col, label=name)
        axes[2].fill_between(tau, 0, t3["mean_rmse_gap"], color=col, alpha=0.10)

    axes[2].axhline(0, color="grey", lw=0.8, ls="--")
    for ax, title, ylab in zip(
        axes,
        ["Clean trajectories", "Event trajectories", "Action-divergence gap"],
        ["Mean HI RMSE", "Mean HI RMSE", "gap = event − clean"],
    ):
        ax.set_xlabel(r"Forecast horizon $\tau$ (steps)")
        ax.set_ylabel(ylab); ax.set_title(title)
        ax.legend(frameon=False)
        ax.xaxis.set_minor_locator(mticker.MultipleLocator(5))

    fig.suptitle("Task 3: Latent Forecasting", y=1.02)
    plt.tight_layout(); plt.show()

In [ ]:
# Task 3 summary table at key horizons
t3_rows = []
for r in all_results:
    if "error" in r or not r.get("task3_forecast"):
        continue
    t3 = r["task3_forecast"]
    for tau in [1, 5, 10, 20, 50]:
        idx = tau - 1
        if idx >= len(t3.get("mean_rmse_all", [])):
            continue
        t3_rows.append({
            "model": label(r["name"]),
            "τ": tau,
            "RMSE (all)":   t3["mean_rmse_all"][idx],
            "RMSE (clean)": t3["mean_rmse_clean"][idx],
            "RMSE (event)": t3["mean_rmse_event"][idx],
            "gap":          t3["mean_rmse_gap"][idx],
        })

if t3_rows:
    display(pd.DataFrame(t3_rows).set_index(["model", "τ"]).round(5))
else:
    print("No Task 3 results")

## Task 1 — R² vs seq_len (all models)

In [ ]:
if not t1_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
    prop_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]

    for metric, ax in zip(["R²", "RMSE", "Pearson-r"], axes):
        for i, m in enumerate(sorted(t1_df["model"].unique())):
            sub = t1_df[t1_df["model"] == m].sort_values("seq_len")
            ax.plot(sub["seq_len"], sub[metric], marker="o",
                    color=prop_cycle[i % len(prop_cycle)], label=m)
        ax.set_title(f"Task 1 — {metric}")
        ax.set_xlabel("seq_len")
        ax.legend(frameon=False)

    plt.tight_layout(); plt.show()

In [ ]:
# ── Per-context heatmap: KL divergence (healthy ‖ degraded) per sensor × context ──
# Shows WHICH contexts carry the most degradation signal.
from scipy.special import rel_entr

N_BINS = 60
n_contexts = sensors_raw.shape[2]

kl_matrix = np.zeros((n_sensors, n_contexts))  # (7, 12)

for s_idx in range(n_sensors):
    for c_idx in range(n_contexts):
        vals_h = sensors_raw[mask_healthy,  s_idx, c_idx]
        vals_d = sensors_raw[mask_degraded, s_idx, c_idx]
        vals_h = vals_h[np.isfinite(vals_h)]
        vals_d = vals_d[np.isfinite(vals_d)]
        if len(vals_h) < 10 or len(vals_d) < 10:
            continue
        lo = min(vals_h.min(), vals_d.min())
        hi = max(vals_h.max(), vals_d.max())
        bins = np.linspace(lo, hi, N_BINS + 1)
        ph, _ = np.histogram(vals_h, bins=bins, density=True)
        pd_, _ = np.histogram(vals_d, bins=bins, density=True)
        # smooth to avoid zeros
        ph  = ph  + 1e-10
        pd_ = pd_ + 1e-10
        ph  = ph  / ph.sum()
        pd_ = pd_ / pd_.sum()
        kl_matrix[s_idx, c_idx] = rel_entr(ph, pd_).sum()

fig, ax = plt.subplots(figsize=(min(n_contexts * 0.7 + 1.5, 14), 3.2))
im = ax.imshow(kl_matrix, aspect="auto", cmap="YlOrRd")
ax.set_yticks(range(n_sensors))
ax.set_yticklabels(names, fontsize=8)
ax.set_xticks(range(n_contexts))
ax.set_xticklabels([f"ctx {c}" for c in range(n_contexts)], fontsize=7, rotation=45, ha="right")
ax.set_title("KL divergence  (healthy ‖ degraded)  per sensor × flight-phase context\n"
             "High value = that sensor/context combination shifts most between health states")
plt.colorbar(im, ax=ax, label="KL div (nats)")
plt.tight_layout()
plt.show()

In [ ]:
# ── KDE plot: healthy vs degraded per sensor ──────────────────────────────────
fig, axes = plt.subplots(1, n_sensors, figsize=(2.5 * n_sensors, 3.5), sharey=False)
if n_sensors == 1:
    axes = [axes]

colors = {"Healthy": "#2196F3", "Degraded": "#F44336"}

for i, (ax, name) in enumerate(zip(axes, names)):
    for label_str, mask, col in [
        ("Healthy",  mask_healthy,  colors["Healthy"]),
        ("Degraded", mask_degraded, colors["Degraded"]),
    ]:
        vals = sensors[mask, i]
        vals = vals[np.isfinite(vals)]
        if len(vals) < 10:
            continue
        lo, hi = np.percentile(vals, 1), np.percentile(vals, 99)
        xs = np.linspace(lo, hi, 300)
        try:
            kde = gaussian_kde(vals, bw_method="scott")
            ax.plot(xs, kde(xs), color=col, label=label_str, lw=1.8)
            ax.fill_between(xs, kde(xs), alpha=0.15, color=col)
        except Exception:
            ax.hist(vals, bins=40, color=col, alpha=0.4, density=True, label=label_str)

    ax.set_title(name, fontsize=8)
    ax.set_xlabel("normalised value")
    ax.tick_params(labelsize=7)
    if i == 0:
        ax.set_ylabel("density")

handles = [plt.Line2D([0], [0], color=c, lw=2, label=l) for l, c in colors.items()]
fig.legend(handles=handles, loc="upper right", frameon=False, fontsize=9)
fig.suptitle(f"Sensor distributions — Healthy vs Degraded  ({ctx_label})", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import h5py
from scipy.stats import gaussian_kde

# ── Config ────────────────────────────────────────────────────────────────────
HDF5_PATH = f"{STABLEWM_HOME}/scenario3_train_400_lewm.h5"

# HI percentile thresholds: top Q% = healthy, bottom Q% = degraded
HEALTHY_QUANTILE  = 0.75   # top 25%
DEGRADED_QUANTILE = 0.25   # bottom 25%

# Which flight-phase context to visualise (0–11). None = average over all 12.
CONTEXT_IDX = None

# Sensor names (7 physical measurements per context in scenario3)
SENSOR_NAMES = [
    "T2 (fan inlet temp)",
    "P2 (fan inlet pres)",
    "T24 (LPC outlet temp)",
    "P24 (LPC outlet pres)",
    "T30 (HPC outlet temp)",
    "Nf (fan speed)",
    "Nc (core speed)",
]

# HI component keys stored in the HDF5
HI_KEYS = [
    "deg_CmpBst_s_mapEff_in", "deg_CmpBst_s_mapWc_in",
    "deg_CmpFan_s_mapEff_in", "deg_CmpFan_s_mapWc_in",
    "deg_CmpH_s_mapEff_in",   "deg_CmpH_s_mapWc_in",
    "deg_TrbH_s_mapEff_in",   "deg_TrbH_s_mapWc_in",
    "deg_TrbL_s_mapEff_in",   "deg_TrbL_s_mapWc_in",
]

# ── Load ─────────────────────────────────────────────────────────────────────
print(f"Loading {HDF5_PATH} ...")
with h5py.File(HDF5_PATH, "r") as f:
    sensors_raw = f["pixels"][:]          # (N, 7, 12)  normalised sensor data
    hi_arrays   = np.stack([f[k][:] for k in HI_KEYS if k in f], axis=1)  # (N, n_hi)

print(f"Loaded {len(sensors_raw):,} timesteps, {hi_arrays.shape[1]} HI components")

# ── Split by mean HI ──────────────────────────────────────────────────────────
mean_hi = hi_arrays.mean(axis=1)          # (N,)  scalar health score per timestep
thr_hi  = np.quantile(mean_hi, HEALTHY_QUANTILE)
thr_lo  = np.quantile(mean_hi, DEGRADED_QUANTILE)

mask_healthy  = mean_hi >= thr_hi
mask_degraded = mean_hi <= thr_lo
print(f"Healthy  (HI ≥ {thr_hi:.3f}): {mask_healthy.sum():,} steps")
print(f"Degraded (HI ≤ {thr_lo:.3f}): {mask_degraded.sum():,} steps")

# ── Aggregate sensors over contexts ──────────────────────────────────────────
if CONTEXT_IDX is not None:
    sensors = sensors_raw[:, :, CONTEXT_IDX]   # (N, 7)
    ctx_label = f"context {CONTEXT_IDX}"
else:
    sensors = sensors_raw.mean(axis=2)         # (N, 7) — average over 12 contexts
    ctx_label = "avg over contexts"

n_sensors = sensors.shape[1]
names = SENSOR_NAMES[:n_sensors] if SENSOR_NAMES else [f"sensor_{i}" for i in range(n_sensors)]

## Sensor Distribution — Healthy vs Degraded
Load raw sensor data directly from the HDF5 file and compare distributions at different HI levels.
"Healthy" = top quartile of mean HI across all components; "Degraded" = bottom quartile.